# A09: BAGGING

In [25]:
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
import numpy as np
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [26]:
df = pd.read_csv("Default.csv")

df["default"] = df["default"].map({"No": 0, "Yes": 1})
df["student"] = df["student"].map({"No": 0, "Yes": 1})


df.head(2)

,default,student,balance,income
0,0,0,729.526495,44361.625074
1,0,1,817.180407,12106.134700


<div style="background-color: #cce5ff; padding: 15px; border-radius: 8px;">
  <h3> AUC-ROC LOGISTIC REGRESSION</h3>
</div>


In [41]:
df = pd.read_csv("Default.csv")

df["default"] = df["default"].map({"No": 0, "Yes": 1})
df["student"] = df["student"].map({"No": 0, "Yes": 1})

target = "default"
columnas = [c for c in df.columns if c != target]

X = df[columnas]
y = df[target]


# Entrenar modelo de regresión logística
modelo = LogisticRegression(solver='liblinear')
modelo.fit(X, y)

# Probabilidad predicha de default

df["prob_default_logit"] = modelo.predict_proba(X)[:, 1]

# Predicción final como clase 0/1
umbral = 0.5
df["pred_default_logit"] = (df["prob_default_logit"] >= umbral).astype(int)


# Calcular métricas

auc = roc_auc_score(y, df["prob_default_logit"])
promedio_global = df["prob_default_logit"].mean()

# Conteo de predicciones
conteo = df["pred_default_logit"].value_counts()

# Moda de las predicciones
moda = df["pred_default_logit"].mode()[0]


# Resultados
print("AUC-ROC Regresión Logística:", auc)
print("Promedio final de probabilidad de default:", promedio_global)
print("\nConteo de predicciones:")
print(conteo)
print("\nModa de las predicciones:", moda)

AUC-ROC Regresión Logística: 0.5950792004376364
Promedio final de probabilidad de default: 0.05719304581824492

Conteo de predicciones:
pred_default_logit
0    9997
1       3
Name: count, dtype: int64

Moda de las predicciones: 0


<div style="background-color: #cce5ff; padding: 15px; border-radius: 8px;">
  <h3> AUC ROC BAGGING</h3>
</div>


In [46]:
# Datos
df = pd.read_csv("Default.csv")
df["default"] = df["default"].map({"No": 0, "Yes": 1})
df["student"] = df["student"].map({"No": 0, "Yes": 1})


# Parámetros bagging

N_MODELOS = 5000
BOOTSTRAP_SIZE = 5000
target = "default"
columnas = [c for c in df.columns if c != target]


# Primer ciclo: entrenar todos los modelos

modelos_guardados = []  # Lista de diccionarios: {'modelo': , 'columnas': }

for i in range(N_MODELOS):
    # Bootstrap
    df_boot = resample(df, n_samples=BOOTSTRAP_SIZE, replace=True, random_state=i)
    
    # Seleccionar 2 columnas aleatorias
    cols_aleatorias = np.random.choice(columnas, size=2, replace=False)
    
    # Entrenar árbol
    X_boot = df_boot[cols_aleatorias]
    y_boot = df_boot[target]
    
    modelo = DecisionTreeClassifier(random_state=i)
    modelo.fit(X_boot, y_boot)
    
    # Guardar el modelo y las columnas usadas
    modelos_guardados.append({'modelo': modelo, 'columnas': cols_aleatorias})


# Segundo ciclo: predecir con cada modelo
probabilidades_modelos = []

for entry in modelos_guardados:
    modelo = entry['modelo']
    cols = entry['columnas']
    prob = modelo.predict_proba(df[cols])[:, 1]
    probabilidades_modelos.append(prob)

# Convertir a matriz
probabilidades_modelos = np.array(probabilidades_modelos)

# Promedio final
pred_final = probabilidades_modelos.mean(axis=0)

# Calcular métricas
auc = roc_auc_score(df[target], pred_final)
promedio_global = pred_final.mean()

# Crear DataFrame final
df_resultado = df.drop(columns=[target]).copy()
df_resultado["prob_default_bagging"] = pred_final

# Predicción final como clase 0/1
umbral = 0.5
df_resultado["pred_default_bagging"] = (pred_final >= umbral).astype(int)

# Mostrar resultados
print("AUC-ROC del modelo Bagging:", auc)
print("Promedio final de probabilidad de default:", promedio_global)
print("\nConteo de predicciones:")
print(df_resultado["pred_default_bagging"].value_counts())
print("\nModa de predicciones:", df_resultado["pred_default_bagging"].mode()[0])


AUC-ROC del modelo Bagging: 0.9999616353707592
Promedio final de probabilidad de default: 0.033459960000000004

Conteo de predicciones:
pred_default_bagging
0    9827
1     173
Name: count, dtype: int64

Moda de predicciones: 0


<div style="background-color: #cce5ff; padding: 15px; border-radius: 8px;">
  <h3> COMPARACION</h3>
</div>


In [49]:
# Crear diccionario con los resultados
resultados = {
    "Modelo": ["Regresión Logística", "Bagging Árboles"],
    "AUC-ROC": [0.5950, 0.9999],
    "Promedio_prob_default": ["5.7193$", "3.3459%"],
    "Conteo_0": [9997, 9827],
    "Conteo_1": [3, 173],
    "Moda": [0, 0]
}

# Crear DataFrame y poner Modelo como índice
df_comparativo = pd.DataFrame(resultados).set_index("Modelo")

# Mostrar
df_comparativo

,AUC-ROC,Promedio_prob_default,Conteo_0,Conteo_1,Moda
Modelo,,,,,
Regresión Logística,0.5950,5.7193$,9997,3,0
Bagging Árboles,0.9999,3.3459%,9827,173,0


<div style="background-color: #cce5ff; padding: 15px; border-radius: 8px;">
  <h3> CONCLUSIONES</h3>
</div>


El bagging de árboles funciona mucho mejor que la regresión logística: identifica casi perfectamente quién hará default y quién no, prediciendo una probabilidad promedio cercana a la tasa real de defaults (3.3%), mientras que la regresión logística apenas discrimina a los clientes de riesgo y sobreestima la probabilidad promedio (5.7%). Ambos modelos predicen mayoritariamente “no default”, pero el bagging detecta muchos más casos de riesgo, por lo que es más confiable y útil para decisiones sobre clientes con probabilidad de default.